# 🔬 TSM — Text Similarity Maker

Generate embeddings of your documents' titles and abstracts, then build a science map.

**Instructions:**
1. Run **Step 1** to install dependencies (~2 min, only needed once per session)
2. Run **Step 2** to launch the app
3. The app will appear below — use it just like the web version

> Built by [Juan Pablo Bascur](https://jpbascur.com)

In [ ]:
#@title Step 1 — Install dependencies (run once) { display-mode: "form" }
%%capture
!pip install streamlit transformers adapters torch umap-learn numpy pandas psutil
!git clone https://github.com/jpbascur/text-similarity-maker.git /content/tsm 2>/dev/null || git -C /content/tsm pull
print('✅ Done. Run Step 2 to launch the app.')

In [ ]:
#@title Step 2 — Launch app { display-mode: "form" }
import subprocess, time, re, urllib.request
from IPython.display import display, IFrame

# Install cloudflared
subprocess.run(
    ['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
     '-O', '/usr/local/bin/cloudflared'], check=True
)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Start Streamlit
subprocess.Popen(
    ['streamlit', 'run', '/content/tsm/streamlit_app.py',
     '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Wait until Streamlit is actually ready
print('Starting Streamlit…')
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        break
    except:
        time.sleep(1)
else:
    print('⚠️ Streamlit took too long to start.')

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url = None
for line in tunnel.stdout:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        break

print(f'✅ App running at: {url}')
print('The app will open below. It may take a few seconds to load.')
display(IFrame(url, width='100%', height=800))